# SFT results

Supervised finetuning on the 27M TinyStories base (`tinystories_2026-09-22_13-20-13`,
val 1.3986). Every number is read back from a run's `artifacts/logs/*.jsonl`.

    python basic.py sft --task reverse  --lr 1e-4
    python basic.py sft --task instruct --lr 1e-4


## Setup


In [1]:
%load_ext autoreload
%autoreload 2

import json
import math
import sys
from pathlib import Path

root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
sys.path.append(str(root / "src"))
sys.path.append(str(root / "src" / "video"))

import torch

from checkpoint import load_checkpoint
from dataset import BinDataset
from evaluate import estimate_loss
from generate import generate
from instruct import Instruct
from lora import load_lora, param_counts
from paths import CKPT_DIR, LOG_DIR
from reverse import Reverse
from tokenizer import ENDOFTEXT

DEV = "cuda" if torch.cuda.is_available() else "cpu"

# a run's jsonl is the archive; a cell re-reads it instead of re-training.
# log stems trail their checkpoint by a second -- the ckpt path is made first.
RUNS = {
    # reverse, 27M TinyStories base
    "rev sft 1e-4": "sft_reverse_2026-09-22_15-46-43",
    "rev lora 1e-4": "lora_reverse_2026-09-22_16-33-47",
    "rev lora 1e-3": "lora_reverse_2026-09-22_16-29-04",
    "rev lora 3e-3": "lora_reverse_2026-09-22_16-20-30",
    # instruct
    "ins sft 3e-5": "sft_instruct_2026-09-22_12-57-54",
    "ins sft 1e-4": "sft_instruct_2026-09-22_17-53-47",
    "ins lora 3e-3": "lora_instruct_2026-09-22_12-26-52",
    "joint lora 1e-2": "lora_joint_2026-09-22_12-47-06",
    # 600-step lr sweeps, eval_n 8 -- only comp is readable at that n
    "sw ins sft 3e-5": "sweep_sft_2026-09-22_17-27-02",
    "sw ins sft 1e-4": "sweep_sft_2026-09-22_17-29-08",
    "sw ins sft 3e-4": "sweep_sft_2026-09-22_17-31-13",
    "sw ins lora 1e-3": "sweep_lora_2026-09-22_17-33-17",
    "sw ins lora 3e-3": "sweep_lora_2026-09-22_17-35-12",
    "sw ins lora 1e-2": "sweep_lora_2026-09-22_17-37-07",
    "sw joint 3e-4": "joint_lr_sweep_2026-09-22_12-13-42",
    "sw joint 1e-3": "joint_lr_sweep_2026-09-22_12-20-41",
    "sw joint 3e-3": "joint_lr_sweep_2026-09-22_12-27-02",
    "sw joint 1e-2": "joint_lr_sweep_2026-09-22_12-34-52",
    "sw rev lora 3e-4": "lora_lr_probe_2026-09-22_16-14-25",
    "sw rev lora 1e-3": "lora_lr_probe_2026-09-22_16-15-01",
    "sw rev lora 3e-3": "lora_lr_probe_2026-09-22_16-15-35",
}
BASE = CKPT_DIR / "tinystories_2026-09-22_13-20-13.pt"
CKPTS = {
    "rev sft 1e-4": CKPT_DIR / "sft_reverse_2026-09-22_15-46-43.pt",
    "rev lora 3e-3": CKPT_DIR / "lora_reverse_2026-09-22_16-20-30.pt",
    "ins sft 3e-5": CKPT_DIR / "sft_instruct_2026-09-22_12-57-53.pt",
    "ins sft 1e-4": CKPT_DIR / "sft_instruct_2026-09-22_17-53-47.pt",
    "ins lora 3e-3": CKPT_DIR / "lora_instruct_2026-09-22_12-26-51.pt",
    "joint lora 1e-2": CKPT_DIR / "lora_joint_2026-09-22_12-47-02.pt",
}


def load_run(label):
    """(config, eval rows, summary)."""
    text = (LOG_DIR / f"{RUNS[label]}.jsonl").read_text()
    ev = [json.loads(line) for line in text.splitlines()]
    pick = lambda e: [x for x in ev if x["event"] == e]
    return pick("config")[0], pick("log"), pick("summary")[0]


def table(rows, cols, w=12):
    print("".join(f"{c:>{w}}" for c in cols))
    for r in rows:
        print(
            "".join(
                f"{v:>{w}.4f}"
                if isinstance(v, float)
                else f"{'-' if v is None else v:>{w}}"
                for v in (r.get(c) for c in cols)
            )
        )

## 1. does the loop work? reverse

`"cat>" -> "tac<eot>"`, scored by string equality on 200 fresh words.
**Reads a log, < 1 s.**

Result: **1.000 by step 300, 113 s.** comp 8.11 -> 0.0000.


In [2]:
cfg, rows, summ = load_run("rev sft 1e-4")
print(f"lr {cfg['lr']:.0e}   window {cfg['window']}   {summ['total_time_s']:.0f}s")
table(rows, ["step", "comp", "prompt", "exact_match", "stop_rate"])

lr 1e-04   window 256   113s
        step        comp      prompt exact_match   stop_rate
           0      8.1094      9.6969      0.0000      0.0050
         100      0.2423      9.5437      0.4200      0.4500
         200      0.0096      9.5719      0.9900      1.0000
         300      0.0024      9.6969      1.0000      1.0000
         400      0.0023      9.9344      1.0000      1.0000
         500      0.0004     10.2531      1.0000      1.0000
         600      0.0003     10.3281      1.0000      1.0000
         700      0.0001      9.2156      1.0000      1.0000
         800      0.0000     10.1781      1.0000      1.0000
         900      0.0000     10.0875      1.0000      1.0000
        1000      0.0000     10.2937      1.0000      1.0000


## 2. what masking costs the prompt

The prompt is masked out of the loss, so its loss is free to move. It rises --
the model specialises and becomes confidently wrong there. **Reads logs, < 1 s.**

Read a starting loss against **ln(V)**, not across tokenizers: ln(4097) = 8.32
against ln(50259) = 10.82.

Result: this base starts **above** uniform on the prompt (+1.38), i.e. worse
than guessing. A story-only model is certain `qxzv` cannot happen.


In [3]:
cfg, rows, _ = load_run("rev sft 1e-4")
lnv = math.log(cfg["vocab_size"])
print(f"ln(V) = ln({cfg['vocab_size']}) = {lnv:.2f}")
print(
    f"step 0:  comp {rows[0]['comp']:.2f} ({rows[0]['comp'] - lnv:+.2f} vs ln V)"
    f"   prompt {rows[0]['prompt']:.2f} ({rows[0]['prompt'] - lnv:+.2f} vs ln V)"
)

solved = next(r["step"] for r in rows if r["exact_match"] == 1.0)
after = [r for r in rows if r["step"] > solved]
lo, hi = min(after, key=lambda r: r["prompt"]), max(after, key=lambda r: r["prompt"])
print(
    f"\nprompt ends {rows[-1]['prompt']:.2f}, but scatters"
    f" {hi['prompt'] - lo['prompt']:.2f} nats after step {solved}"
    f" on identical val windows -- read it as 'about 10.2', not a trend."
)

ln(V) = ln(4097) = 8.32
step 0:  comp 8.11 (-0.21 vs ln V)   prompt 9.70 (+1.38 vs ln V)

prompt ends 10.29, but scatters 1.11 nats after step 300 on identical val windows -- read it as 'about 10.2', not a trend.


## 3. the real task: TinyStoriesInstruct

Fields in, a story out. lr 1e-4, measured. **Reads a log, < 1 s.**

Result: **comp 1.5043 -> 1.0945, words_all 0.00 -> 0.42, stop_rate 0.58 -> 0.93.**

This base stops stories on its own already; what SFT buys here is the _format_
and the word constraints, not termination. Stories grow 56 -> 153 words, toward
the corpus mean.


In [4]:
cfg, rows, summ = load_run("ins sft 1e-4")
print(
    f"lr {cfg['lr']:.0e}   window {cfg['window']}   {summ['total_time_s'] / 60:.1f} min"
)
table(
    rows, ["step", "comp", "prompt", "words", "words_all", "stop_rate", "story_words"]
)
best = min(rows, key=lambda r: r["comp"])
print(
    f"\nbest comp {best['comp']:.4f} @ step {best['step']} -- the kept checkpoint"
    f" (--select comp; selecting on `words` keeps a noisier, worse one)"
)

lr 1e-04   window 512   9.4 min
        step        comp      prompt       words   words_all   stop_rate story_words
           0      1.5043      4.0922      0.1250      0.0000      0.5750     56.4750
         200      1.1477      4.6594      0.7000      0.2250      0.8750    154.3750
         400      1.1342      4.7391      0.6917      0.3250      0.8750    150.7750
         600      1.1240      4.6828      0.7750      0.4000      0.9500    149.4750
         800      1.1156      4.6516      0.7500      0.4000      0.8500    156.8750
        1000      1.1105      4.7047      0.7333      0.4000      0.8750    150.4000
        1200      1.1043      4.7016      0.7500      0.3750      0.8750    151.1750
        1400      1.1014      4.6969      0.7500      0.4000      0.8500    151.4250
        1600      1.0973      4.6875      0.7500      0.3750      0.9000    154.4750
        1800      1.0945      4.6891      0.7333      0.4000      0.8500    152.8250
        2000      1.0945      4.6

## 4. what it writes

Three held-out prompts, greedy and two samples at t=0.8 p=0.95.
**Loads a checkpoint, ~1 min.**

Result: **6 of 9 samples get every required word**; stopping is now the weak
axis (5 of 9), and the non-stoppers are the long ones.


In [5]:
task = Instruct()
model, meta = load_checkpoint(CKPTS["ins sft 1e-4"], DEV)
model.eval()
print(f"step {meta['step']}   comp {meta['val_loss']:.4f}\n")

for i in (1, 3, 7):
    prompt = task.prompts()[i]
    x = torch.tensor([task.tok.encode(prompt)], device=DEV)
    print("=" * 76)
    print(prompt)
    for tag, kw in (
        ("greedy", {"temperature": 0.0}),
        ("t=0.8 #0", {"temperature": 0.8, "top_p": 0.95, "seed": 0}),
        ("t=0.8 #1", {"temperature": 0.8, "top_p": 0.95, "seed": 1}),
    ):
        kw = dict(kw)
        seed = kw.pop("seed", 0)
        if kw["temperature"]:
            kw["generator"] = torch.Generator(device=DEV).manual_seed(seed)
        out = generate(model, x, 300, use_cache=True, **kw)
        text = task.tok.decode(out[0, x.size(1) :].tolist())
        story = text.split(ENDOFTEXT)[0]
        print(
            f"\n--- {tag} | stopped {ENDOFTEXT in text}"
            f" | reward {task.reward(prompt, text):.2f}"
            f" | {len(story.split())} words ---"
        )
        print(story)
    print()

step 1800   comp 1.0945

Words: meet, waffle, new
Summary: Lily wants a waffle but gets bitten by a dog while playing with a toy car and has to go to the hospital for stitches.
Story:


--- greedy | stopped True | reward 0.64 | 178 words ---
Once upon a time, there was a little girl named Lily. She loved waffles and ate them every day. One day, she went to the park to play with her friends. While she was playing, she saw a new toy car. She wanted it so badly, but she didn't have one.
Suddenly, a big dog came running towards her. Lily got scared and ran away. She didn't know what to do. Then, she saw a man with a toy car. She went to him and asked for a waffle. The man gave her a new toy car and Lily was happy again.
But then, she saw a dog running towards her. She got scared again and ran away. The man tried to catch her, but she was too fast. She ran and ran until she was safe. She realized that the new toy car was just a toy and she didn't need it. She went home and told her mom abou